In [1]:
#CTEs use karke complex chained query ko easy banana.

In [2]:
#isme hum multiple CTEs ko chain karenge - matlab ek CTE dusre CTE ke result par based hogi(jaise step-by-step pipeline). yeh pattern aage RFM analysis (month 5, day 81) banane ke liye directly kaam aaiga - isliye is din ko dhyan se karna.

In [3]:
#01: 3-step chained CTE-recency + frequency + monetary ek saath (mini-RFM preview):

In [4]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

q1 = pd.read_sql("""
    WITH customer_orders AS (
        -- Step 1: Har customer ke saare orders ki basic info
        SELECT "Customer ID", Invoice, InvoiceDate, OrderValue
        FROM orders
    ),
    customer_summary AS (
        -- Step 2: Step 1 ki CTE use karke summary banao (Frequency + Monetary)
        SELECT "Customer ID",
               COUNT(Invoice) as frequency,
               SUM(OrderValue) as monetary,
               MAX(InvoiceDate) as last_order_date
        FROM customer_orders
        GROUP BY "Customer ID"
    ),
    customer_recency AS (
        -- Step 3: Step 2 ki CTE use karke Recency nikalo (last order se aaj tak ka gap)
        SELECT "Customer ID", frequency, monetary, last_order_date,
               julianday('2011-12-10') - julianday(last_order_date) as recency_days
        FROM customer_summary
    )
    SELECT * FROM customer_recency
    ORDER BY recency_days DESC
    LIMIT 15
""", conn)
print(q1)

    Customer ID  frequency  monetary      last_order_date  recency_days
0       12636.0          1    141.00  2009-12-01 09:55:00    738.586806
1       17592.0          2      0.00  2009-12-01 10:49:00    738.549306
2       17641.0          1     -6.95  2009-12-01 12:35:00    738.475694
3       17056.0          1    128.60  2009-12-01 12:55:00    738.461806
4       14654.0          1    246.86  2009-12-01 12:57:00    738.460417
5       13526.0          4    942.60  2009-12-01 13:14:00    738.448611
6       17485.0          1    -29.55  2009-12-01 14:52:00    738.380556
7       17087.0          1    221.53  2009-12-02 10:41:00    737.554861
8       17818.0          1    130.18  2009-12-02 11:34:00    737.518056
9       15833.0          1     80.40  2009-12-02 11:59:00    737.500694
10      17909.0          1    132.55  2009-12-02 13:10:00    737.451389
11      17606.0          1     87.30  2009-12-02 13:32:00    737.436111
12      14106.0          1    214.80  2009-12-02 14:09:00    737

In [5]:
# 3 CTEs chain ho rahi hai - customer_orders -> customer_summary -> customer_recency. Har step pichle step ke result ko aage badhata hai, exactly jaise ek data pipeline kaam karta hai. yeh Bouns day 35B ke churn-label logic ka SQL-side foundation hai.

In [6]:
#02: Chained CTE + filtering - high-value customers search karo jo recently active nahi hai(early churn-risk signal):

In [7]:
q2 = pd.read_sql("""
    WITH customer_summary AS (
        SELECT "Customer ID",
               COUNT(Invoice) as frequency,
               SUM(OrderValue) as monetary,
               MAX(InvoiceDate) as last_order_date
        FROM orders
        GROUP BY "Customer ID"
    ),
    customer_recency AS (
        SELECT "Customer ID", frequency, monetary,
               julianday('2011-12-10') - julianday(last_order_date) as recency_days
        FROM customer_summary
    )
    SELECT * FROM customer_recency
    WHERE monetary > 2000 AND recency_days > 90
    ORDER BY monetary DESC
""", conn)
print(q2)
print(f"High-value at-risk customers: {q2.shape[0]}")

     Customer ID  frequency  monetary  recency_days
0        16754.0         35  56560.58    372.265278
1        17850.0        159  55703.13    302.390278
2        13093.0         88  54073.73    267.280556
3        13902.0          8  30411.26    632.452083
4        13802.0         24  25491.56    138.440278
..           ...        ...       ...           ...
326      14225.0          5   2015.12    434.561111
327      15384.0          5   2014.49    169.543750
328      13663.0          6   2013.61    179.344444
329      16529.0          4   2012.00    190.275694
330      13696.0          4   2008.70    421.594444

[331 rows x 4 columns]
High-value at-risk customers: 331


In [8]:
#yeh exactly wahi customers hai jin per bussiness ko sabse pehle focus karna chaiye - high spend but active.

In [9]:
#03:Chained CTE + window function combine karke - customer ko unke monetary quartile ke saath dikhana:

In [10]:
q3 = pd.read_sql("""
    WITH customer_summary AS (
        SELECT "Customer ID", SUM(OrderValue) as monetary
        FROM orders
        GROUP BY "Customer ID"
    ),
    customer_segments AS (
        SELECT "Customer ID", monetary,
               NTILE(4) OVER (ORDER BY monetary DESC) as value_quartile
        FROM customer_summary
    )
    SELECT value_quartile, COUNT(*) as num_customers, AVG(monetary) as avg_monetary
    FROM customer_segments
    GROUP BY value_quartile
    ORDER BY value_quartile
""", conn)
print(q3)

   value_quartile  num_customers  avg_monetary
0               1           1486   9173.360156
1               2           1486   1376.143421
2               3           1485    546.680008
3               4           1485    107.683677


In [11]:
#practice questions.

In [12]:
#1.Ek 4th CTE add karo upar wale Step 1 example mein — jo frequency, monetary, aur recency_days teeno ko combine karke ek simple "score" banaye (jaise monetary/recency_days), aur top 10 highest-score customers dikhaye.

In [13]:
#

In [14]:
#2.Chained CTE se pata karo: kaunse countries mein "high recency_days" (matlab inactive) customers sabse zyada hain.

In [15]:
#

In [16]:
conn.close()